# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [6]:
#Clé unique par ligne
from pyspark.sql.functions import monotonically_increasing_id
df_trips = df_trips.withColumn("trip_id", monotonically_increasing_id())

In [7]:
#passager max
df_trips.orderBy(df_trips.passenger_count.desc()).show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_id|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+
|       2| 2019-01-05 13:12:29|  2019-01-05 13:12:32|            9.0|          0.0|       5.0|                 N|          68|          68|           1|        9.8

In [8]:
#moyenne passager
from pyspark.sql.functions import avg
df_trips.select(avg("passenger_count")).show()

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+



In [9]:
#trip le court/long
from pyspark.sql.functions import col, unix_timestamp
df_trips = df_trips.withColumn(
    "trip_duration_min",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)
df_trips.orderBy(col("trip_distance").asc()).show(1)
df_trips.orderBy(col("trip_distance").desc()).show(1)
df_trips.orderBy(col("trip_duration_min").asc()).show(1)
df_trips.orderBy(col("trip_duration_min").desc()).show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_id|trip_duration_min|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+-----------------+
|       2| 2018-12-21 13:48:30|  2018-12-21 13:52:40|            3.0|          0.0|       1.0|               

In [10]:
#busiest day/slowest single day
from pyspark.sql.functions import to_date
df_trips.withColumn("pickup_date", to_date("tpep_pickup_datetime")) \
    .groupBy("pickup_date").count().orderBy(col("count").desc()).show(5)

+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|292499|
| 2019-01-11|291714|
| 2019-01-31|284625|
| 2019-01-17|284580|
| 2019-01-24|281959|
+-----------+------+
only showing top 5 rows


In [11]:
#busiest/slowest time of day
from pyspark.sql.functions import hour, when
df_trips = df_trips.withColumn("pickup_hour", hour("tpep_pickup_datetime"))
df_trips = df_trips.withColumn(
    "time_bucket",
    when(col("pickup_hour").between(5, 11), "morning")
    .when(col("pickup_hour").between(12, 16), "afternoon")
    .when(col("pickup_hour").between(17, 21), "evening")
    .otherwise("late_night")
)
df_trips.groupBy("time_bucket").count().orderBy(col("count").desc()).show()

+-----------+-------+
|time_bucket|  count|
+-----------+-------+
|    evening|2292112|
|  afternoon|2111999|
|    morning|2035497|
| late_night|1257009|
+-----------+-------+



In [12]:
#On average which day of the week is slowest/busiest
from pyspark.sql.functions import dayofweek
df_trips = df_trips.withColumn("day_of_week", dayofweek("tpep_pickup_datetime"))
df_trips.groupBy("day_of_week").count().orderBy(col("count").desc()).show()

+-----------+-------+
|day_of_week|  count|
+-----------+-------+
|          5|1357043|
|          4|1265264|
|          3|1209084|
|          6|1087215|
|          7|1009985|
|          2| 908121|
|          1| 859905|
+-----------+-------+



In [14]:
from pyspark.sql.functions import corr

#distance passager pourboire
df_trips.select(
    corr("trip_distance", "tip_amount"),
    corr("passenger_count", "tip_amount")
).show()

+-------------------------------+---------------------------------+
|corr(trip_distance, tip_amount)|corr(passenger_count, tip_amount)|
+-------------------------------+---------------------------------+
|             0.5269200663652668|             0.001084223312167...|
+-------------------------------+---------------------------------+



In [15]:
#charge extra élevée
df_trips.orderBy(col("extra").desc()).show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+-----------------+-----------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount| extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_id|trip_duration_min|pickup_hour|time_bucket|day_of_week|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+-----------------+-----------+-----------+-----------

Each trip was given a unique trip_id. On average, trips carried about 1.57 passengers, and the highest passenger count appeared on a clearly erroneous trip with 0 distance and a 3-second duration. Distance and duration values also revealed several outliers: one trip covered 831.8 miles in under 10 minutes (impossible speed), another had a negative duration due to a dropoff timestamp earlier than pickup, and another lasted nearly 30 days, likely from a meter never closed out properly. In terms of demand, January 25th, 2019 was the busiest day, evenings (5–9 PM) were the busiest time of day, and Thursdays were the busiest day of the week, while Sundays were the slowest. Trip distance showed a moderate positive correlation with tip amount (≈0.53), while passenger count had almost no effect (≈0.001). The highest extra charge 535.38 dollars. Overall, these zero/negative/extreme values across distance, duration, and charges appear to be sensor or entry errors and should be filtered out before computing reliable averages.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [16]:
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
response = requests.get(zone_url)

zone_file = "taxi_zone_lookup.csv"
if response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(response.content)

df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(zone_file)

df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [17]:
#join
df_trips_with_zones = df_trips \
    .join(df_zones.withColumnRenamed("LocationID", "PULocationID")
                  .withColumnRenamed("Borough", "PU_Borough"),
          on="PULocationID", how="left") \
    .join(df_zones.withColumnRenamed("LocationID", "DOLocationID")
                  .withColumnRenamed("Borough", "DO_Borough"),
          on="DOLocationID", how="left")

In [18]:
#+ de pickup et drop
df_trips_with_zones.groupBy("PU_Borough").count().orderBy(col("count").desc()).show()
df_trips_with_zones.groupBy("DO_Borough").count().orderBy(col("count").desc()).show()

+-------------+-------+
|   PU_Borough|  count|
+-------------+-------+
|    Manhattan|6950965|
|       Queens| 471173|
|      Unknown| 159815|
|     Brooklyn|  91905|
|        Bronx|  18062|
|          N/A|   3890|
|          EWR|    446|
|Staten Island|    361|
+-------------+-------+

+-------------+-------+
|   DO_Borough|  count|
+-------------+-------+
|    Manhattan|6817355|
|       Queens| 340972|
|     Brooklyn| 301105|
|      Unknown| 149097|
|        Bronx|  58085|
|          N/A|  16904|
|          EWR|  10914|
|Staten Island|   2185|
+-------------+-------+



Which borough had the most pickups? dropoffs?
Manhattan had by far the most pickups (6,950,965) and the most dropoffs (6,817,355), followed by Queens for both. This makes sense given Manhattan's density and its status as the main yellow-taxi service area.

In [19]:
#heure chargée et calme
df_trips_with_zones.groupBy("PU_Borough", "time_bucket").count() \
    .orderBy("PU_Borough", col("count").desc()).show(30)

+-------------+-----------+-------+
|   PU_Borough|time_bucket|  count|
+-------------+-----------+-------+
|        Bronx|    morning|   8407|
|        Bronx|  afternoon|   4541|
|        Bronx|    evening|   2920|
|        Bronx| late_night|   2194|
|     Brooklyn|    morning|  31544|
|     Brooklyn| late_night|  22354|
|     Brooklyn|    evening|  19134|
|     Brooklyn|  afternoon|  18873|
|          EWR|  afternoon|    206|
|          EWR|    morning|    120|
|          EWR|    evening|     95|
|          EWR| late_night|     25|
|    Manhattan|    evening|2080866|
|    Manhattan|  afternoon|1912287|
|    Manhattan|    morning|1836840|
|    Manhattan| late_night|1120972|
|          N/A| late_night|   1082|
|          N/A|    morning|   1008|
|          N/A|    evening|    930|
|          N/A|  afternoon|    870|
|       Queens|    evening| 140624|
|       Queens|  afternoon| 130169|
|       Queens|    morning| 115882|
|       Queens| late_night|  84498|
|Staten Island|    morning| 

What are the busy/slow times by borough?
For Manhattan, Queens, and Brooklyn, evenings were the busiest period, followed by afternoon and morning, with late night being the slowest. The Bronx was the exception: mornings were its busiest period, while late night was the slowest.

In [20]:
#jour le plus chargé
df_trips_with_zones.groupBy("PU_Borough", "day_of_week").count() \
    .orderBy("PU_Borough", col("count").desc()).show(50)

+-------------+-----------+-------+
|   PU_Borough|day_of_week|  count|
+-------------+-----------+-------+
|        Bronx|          5|   3121|
|        Bronx|          3|   3059|
|        Bronx|          4|   2999|
|        Bronx|          6|   2666|
|        Bronx|          2|   2177|
|        Bronx|          1|   2112|
|        Bronx|          7|   1928|
|     Brooklyn|          3|  15779|
|     Brooklyn|          5|  15714|
|     Brooklyn|          4|  15101|
|     Brooklyn|          6|  13092|
|     Brooklyn|          7|  11604|
|     Brooklyn|          1|  11099|
|     Brooklyn|          2|   9516|
|          EWR|          4|     83|
|          EWR|          3|     77|
|          EWR|          6|     74|
|          EWR|          1|     68|
|          EWR|          5|     58|
|          EWR|          7|     55|
|          EWR|          2|     31|
|    Manhattan|          5|1229554|
|    Manhattan|          4|1144782|
|    Manhattan|          3|1086202|
|    Manhattan|          6| 

What are the busiest days of the week by borough?
Manhattan, Brooklyn, and Queens were all busiest on day 5 (Thursday). The Bronx was busiest on day 5 as well but with day 3 (Tuesday) close behind. Across all boroughs, day 7 or day 1 (Sunday/weekend) tended to be among the slowest.

In [21]:
#distance par secteur
df_trips_with_zones.groupBy("PU_Borough").agg(avg("trip_distance")).show()

+-------------+------------------+
|   PU_Borough|avg(trip_distance)|
+-------------+------------------+
|       Queens|11.283218499361993|
|          EWR| 2.641098654708519|
|      Unknown| 2.415464130400774|
|     Brooklyn| 4.787677275447492|
|Staten Island|12.503601108033246|
|          N/A| 3.193850899742941|
|    Manhattan|2.2286693358402596|
|        Bronx| 7.233194552098303|
+-------------+------------------+



What is the average trip distance by borough?
Staten Island had the longest average trip distance (12.5 miles), followed by Queens (11.28 miles), both likely due to trips heading to/from JFK/LaGuardia or longer outer-borough distances. Manhattan had the shortest average distance (2.23 miles), consistent with dense, short in-borough trips.

In [23]:
#tarif par quartier
df_trips_with_zones.groupBy("PU_Borough").agg(avg("fare_amount")).show()

+-------------+------------------+
|   PU_Borough|  avg(fare_amount)|
+-------------+------------------+
|       Queens| 35.14462651722029|
|          EWR| 76.24024663677126|
|      Unknown|14.944423051653523|
|     Brooklyn|18.649132800172286|
|Staten Island|45.289861495844896|
|          N/A|  59.5731593830335|
|    Manhattan|10.792468572351568|
|        Bronx| 26.26890543682963|
+-------------+------------------+



What is the average trip fare by borough? The airport have the highest average trip distance followed by stalen island and queens.Manhattan have the lowest fare amount because of it's short distance.


In [24]:
#tarif max min par quartier
df_trips_with_zones.orderBy(col("fare_amount").desc()).select("PU_Borough", "fare_amount").show(1)
df_trips_with_zones.orderBy(col("fare_amount").asc()).select("PU_Borough", "fare_amount").show(1)

+----------+-----------+
|PU_Borough|fare_amount|
+----------+-----------+
| Manhattan|  623259.86|
+----------+-----------+
only showing top 1 row
+----------+-----------+
|PU_Borough|fare_amount|
+----------+-----------+
|    Queens|     -362.0|
+----------+-----------+
only showing top 1 row


Highest/lowest fare amounts for a trip, and the associated borough?
The highest single fare was Manhattan associate with pickup in queens

In [25]:
#charger data 2025
download_url_2025 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet'

# charger la data
response = requests.get(download_url_2025)

jan_2025_trip_data = "yellow_tripdata_2025-01.parquet"
if response.status_code == 200:
    with open(jan_2025_trip_data, "wb") as f:
        f.write(response.content)

# créer le dataframe
df_trips_2025 = spark.read.parquet(jan_2025_trip_data)

# regarder le schéma
df_trips_2025.printSchema()
df_trips_2025.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

In [26]:
# recréer les mêmes colonnes que sur df_trips (2019)
df_trips_2025 = df_trips_2025.withColumn(
    "trip_duration_min",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)

df_trips_2025 = df_trips_2025.withColumn("pickup_hour", hour("tpep_pickup_datetime"))

df_trips_2025 = df_trips_2025.withColumn(
    "time_bucket",
    when(col("pickup_hour").between(5, 11), "morning")
    .when(col("pickup_hour").between(12, 16), "afternoon")
    .when(col("pickup_hour").between(17, 21), "evening")
    .otherwise("late_night")
)

df_trips_2025 = df_trips_2025.withColumn("day_of_week", dayofweek("tpep_pickup_datetime"))

In [27]:
df_trips_2025_with_zones = df_trips_2025 \
    .join(df_zones.withColumnRenamed("LocationID", "PULocationID")
                  .withColumnRenamed("Borough", "PU_Borough"),
          on="PULocationID", how="left") \
    .join(df_zones.withColumnRenamed("LocationID", "DOLocationID")
                  .withColumnRenamed("Borough", "DO_Borough"),
          on="DOLocationID", how="left")

In [28]:
# pickups / dropoffs par borough
df_trips_2025_with_zones.groupBy("PU_Borough").count().orderBy(col("count").desc()).show()
df_trips_2025_with_zones.groupBy("DO_Borough").count().orderBy(col("count").desc()).show()

# distance moyenne par borough
df_trips_2025_with_zones.groupBy("PU_Borough").agg(avg("trip_distance")).show()

# tarif moyen par borough
df_trips_2025_with_zones.groupBy("PU_Borough").agg(avg("fare_amount")).show()

# fare max/min et borough associé
df_trips_2025_with_zones.orderBy(col("fare_amount").desc()).select("PU_Borough", "fare_amount").show(1)
df_trips_2025_with_zones.orderBy(col("fare_amount").asc()).select("PU_Borough", "fare_amount").show(1)

+-------------+-------+
|   PU_Borough|  count|
+-------------+-------+
|    Manhattan|3089275|
|       Queens| 294986|
|     Brooklyn|  66070|
|        Bronx|  14741|
|      Unknown|   8141|
|          N/A|   1380|
|          EWR|    377|
|Staten Island|    256|
+-------------+-------+

+-------------+-------+
|   DO_Borough|  count|
+-------------+-------+
|    Manhattan|3116979|
|       Queens| 162896|
|     Brooklyn| 140987|
|        Bronx|  22728|
|          N/A|  12086|
|      Unknown|  11976|
|          EWR|   6873|
|Staten Island|    701|
+-------------+-------+

+-------------+------------------+
|   PU_Borough|avg(trip_distance)|
+-------------+------------------+
|       Queens| 13.36087438047892|
|          EWR|0.8962864721485412|
|      Unknown| 3.202056258444911|
|     Brooklyn|24.807041925231132|
|Staten Island| 8.305703124999997|
|          N/A|28.235833333333336|
|    Manhattan| 4.443918890353684|
|        Bronx| 65.91340478936304|
+-------------+------------------+

+

In [29]:
avg_2019 = df_trips_with_zones.groupBy("PU_Borough") \
    .agg(avg("trip_distance").alias("avg_distance_2019"),
         avg("fare_amount").alias("avg_fare_2019"))

avg_2025 = df_trips_2025_with_zones.groupBy("PU_Borough") \
    .agg(avg("trip_distance").alias("avg_distance_2025"),
         avg("fare_amount").alias("avg_fare_2025"))

comparison = avg_2019.join(avg_2025, on="PU_Borough", how="outer")
comparison.show()

+-------------+------------------+------------------+------------------+------------------+
|   PU_Borough| avg_distance_2019|     avg_fare_2019| avg_distance_2025|     avg_fare_2025|
+-------------+------------------+------------------+------------------+------------------+
|        Bronx| 7.233194552098303| 26.26890543682963| 65.91340478936304|27.772212197272854|
|     Brooklyn| 4.787677275447492|18.649132800172286|24.807041925231132| 23.43692280914226|
|          EWR| 2.641098654708519| 76.24024663677126|0.8962864721485412| 81.47989389920424|
|    Manhattan|2.2286693358402596|10.792468572351568| 4.443918890353684|13.869130184269373|
|          N/A| 3.193850899742941|  59.5731593830335|28.235833333333336|  78.3266884057971|
|       Queens|11.283218499361993| 35.14462651722029| 13.36087438047892|48.350709626896915|
|Staten Island|12.503601108033246|45.289861495844896| 8.305703124999997|23.920820312500002|
|      Unknown| 2.415464130400774|14.944423051653523| 3.202056258444911| 18.6691

2019 vs 2025 Comparison

When we compare 2019 and 2025, fare go up in most borough, like Manhattan (10.79 → 13.82) and Queens (35.14 → 48.35), probably because of inflation. But Staten Island fare actually go down (45.29 → 23.92). For distance, some borough like Bronx (7.23 → 65.91 miles) and Brooklyn (4.79 → 24.81 miles) show a huge jump that dont make sense realistically, this is probably outliers/bad GPS data same like we saw in Part 1, not a real change in trips. Manhattan distance also almost double (2.23 → 4.44 miles) which could be part real and part outliers too. So overall, the fare increase look like a real trend, but the distance numbers are probably not reliable and would need to filter the outliers before comparing properly.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [30]:
#vue temporaire
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

In [31]:
#moyenne nombre de passager
spark.sql("""
SELECT AVG(passenger_count) AS avg_passenger_count
FROM trips
""").show()

+-------------------+
|avg_passenger_count|
+-------------------+
| 1.5670317144945614|
+-------------------+



In [32]:
#jour de la semaine chargé pas chargé
spark.sql("""
SELECT DAYOFWEEK(tpep_pickup_datetime) AS day_of_week, COUNT(*) AS trip_count
FROM trips
GROUP BY DAYOFWEEK(tpep_pickup_datetime)
ORDER BY trip_count DESC
""").show()

+-----------+----------+
|day_of_week|trip_count|
+-----------+----------+
|          5|   1357043|
|          4|   1265264|
|          3|   1209084|
|          6|   1087215|
|          7|   1009985|
|          2|    908121|
|          1|    859905|
+-----------+----------+



In [34]:
#quartier avec le plus de pickup
spark.sql("""
SELECT z.Borough AS PU_Borough, COUNT(*) AS pickup_count
FROM trips t
JOIN zones z ON t.PULocationID = z.LocationID
GROUP BY z.Borough
ORDER BY pickup_count DESC
""").show()

+-------------+------------+
|   PU_Borough|pickup_count|
+-------------+------------+
|    Manhattan|     6950965|
|       Queens|      471173|
|      Unknown|      159815|
|     Brooklyn|       91905|
|        Bronx|       18062|
|          N/A|        3890|
|          EWR|         446|
|Staten Island|         361|
+-------------+------------+



For Part 3, I redid 3 questions in pure SQL instead of python: the average passenger count, the busiest/slowest day of the week, and which borough had the most pickups (this last one uses a JOIN between the trips and zones tables). All three queries return the same results as their python equivalents in Part 1 and Part 2, confirming that both approaches are consistent.

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing

In [35]:
#saison la plus chargée !
months_2019 = [f"{m:02d}" for m in range(1, 13)]
dfs = []

for month in months_2019:
    url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-{month}.parquet'
    filename = f"yellow_tripdata_2019-{month}.parquet"
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
        dfs.append(spark.read.parquet(filename))

df_2019_full = dfs[0]
for df in dfs[1:]:
    df_2019_full = df_2019_full.unionByName(df, allowMissingColumns=True)

In [36]:
from pyspark.sql.functions import month

df_2019_full = df_2019_full.withColumn("pickup_month", month("tpep_pickup_datetime"))

df_2019_full = df_2019_full.withColumn(
    "season",
    when(col("pickup_month").isin(12, 1, 2), "winter")
    .when(col("pickup_month").isin(3, 4, 5), "spring")
    .when(col("pickup_month").isin(6, 7, 8), "summer")
    .otherwise("fall")
)

df_2019_full.groupBy("season").count().orderBy(col("count").desc()).show()

+------+--------+
|season|   count|
+------+--------+
|spring|22941027|
|winter|21643025|
|  fall|20659565|
|summer|19354827|
+------+--------+



In [37]:
#par jour de la semaine
df_trips.groupBy("day_of_week").count().orderBy("day_of_week") \
    .plot.bar(x="day_of_week", y="count")

In [38]:
#distance vs pourboire
df_trips.sample(0.001).plot.scatter(x="trip_distance", y="tip_amount")

In [39]:
#nombre de voyage par quartier
df_trips_with_zones.groupBy("PU_Borough").count() \
    .plot.bar(x="PU_Borough", y="count")